Exercise 7.1: "Dask Array Operations on the Cluster"
Connect your Dask client to the Strato cluster scheduler.
Create two large random Dask arrays using dask.array.random.random with a chosen chunk layout.
Compute the matrix–vector product between them on the cluster and measure the wall-clock time.
Repeat for different array sizes and chunk shapes and observe how chunking affects performance.
Use client.who_has() and client.nthreads() to inspect how work is distributed across workers.

(Hint: generate new arrays each time to prevent Dask from reusing cached results.)



In [ ]:
import os
import time
import numpy as np
import dask.array as da
from dask.distributed import Client, get_client, wait

# Set these directly if you want to force Strato from inside the notebook.
# Keep as None to use environment variables instead.
MANUAL_STRATO_HOST = "10.92.1.34"
MANUAL_STRATO_PORT = "8786"
ALLOW_LOCAL_FALLBACK = False


def _build_scheduler_candidates():
    """Return scheduler addresses to try in priority order."""
    manual_host = (MANUAL_STRATO_HOST or "").strip()
    manual_port = (MANUAL_STRATO_PORT or "").strip()
    env_scheduler = os.getenv("STRATO_SCHEDULER_ADDRESS", "").strip()
    env_host = os.getenv("STRATO_HOST", "").strip()
    env_port = os.getenv("STRATO_PORT", "").strip()

    candidates = []
    if manual_host and manual_port:
        candidates.append(f"tcp://{manual_host}:{manual_port}")
    elif env_scheduler:
        candidates.append(env_scheduler)
    elif env_host and env_port:
        candidates.append(f"tcp://{env_host}:{env_port}")

    # If dashboard port 8787 is provided, also try common scheduler port 8786.
    expanded = []
    for addr in candidates:
        expanded.append(addr)
        if addr.endswith(":8787"):
            expanded.append(addr[:-4] + "8786")
    return expanded


def connect_client(force_reconnect=True, allow_local_fallback=ALLOW_LOCAL_FALLBACK):
    """Connect to Strato scheduler if configured, else reuse/create a local client."""
    candidates = _build_scheduler_candidates()

    if candidates:
        if force_reconnect:
            try:
                old = get_client()
                old.close()
                print("Closed existing local/reused client before reconnect")
            except ValueError:
                pass

        errors = []
        for scheduler in candidates:
            try:
                print(f"Connecting to Strato scheduler: {scheduler}")
                return Client(scheduler, timeout="10s")
            except Exception as exc:
                errors.append((scheduler, exc))
                print(f"Strato connection failed for {scheduler}: {exc}")

        if not allow_local_fallback:
            joined = " | ".join([f"{addr}: {err}" for addr, err in errors])
            raise RuntimeError(f"Could not connect to Strato scheduler. Tried: {joined}")

        print("WARNING: Falling back to existing/local Dask client")

    try:
        client = get_client()
        print("Reusing existing Dask client")
        return client
    except ValueError:
        print("No Strato address found; connecting with local Client()")
        return Client()


def benchmark_matvec(n, chunks):
    """Run one matrix-vector benchmark and return timing/result metadata."""
    # Create fresh arrays each run so Dask does not reuse old task graphs/results.
    A = da.random.random((n, n), chunks=chunks)
    x = da.random.random((n,), chunks=(chunks[1],))

    start = time.perf_counter()
    y = (A @ x).persist()
    wait(y)
    checksum = float(y.sum().compute())
    wall = time.perf_counter() - start

    return {
        "n": n,
        "chunks": chunks,
        "wall_time_s": wall,
        "checksum": checksum,
        "num_blocks_A": int(np.prod(A.numblocks)),
    }


client = connect_client(force_reconnect=True)
print(client)
print("Workers and threads:", client.nthreads())

# (n, chunk shape) configurations
experiments = [
    (4000, (1000, 1000)),
    (4000, (2000, 500)),
    (6000, (1500, 1500)),
    (6000, (3000, 750)),
]

results = []
for n, chunks in experiments:
    print(f"\nRunning n={n}, chunks={chunks}...")
    out = benchmark_matvec(n, chunks)
    results.append(out)

    # Inspect global object placement across workers.
    who_has = client.who_has()
    worker_counts = {}
    for workers in who_has.values():
        for w in workers:
            worker_counts[w] = worker_counts.get(w, 0) + 1
    print("Objects tracked per worker:", worker_counts)

print("\nBenchmark summary (sorted by n, then time):")
for row in sorted(results, key=lambda r: (r["n"], r["wall_time_s"])):
    print(
        f"n={row['n']}, chunks={row['chunks']}, "
        f"time={row['wall_time_s']:.4f}s, "
        f"blocks={row['num_blocks_A']}, checksum={row['checksum']:.6f}"
    )

results

Closed existing local/reused client before reconnect
Connecting to Strato scheduler: tcp://10.92.1.34:8786
<Client: 'tcp://10.92.1.34:8786' processes=0 threads=0, memory=0 B>
Workers and threads: {}

Running n=4000, chunks=(1000, 1000)...


In [ ]:
from dask.distributed import Client

addr = "tcp://10.92.1.34:8786"   # use 127.0.0.1:8786 only if you are tunneling
client = Client(addr, timeout="10s")

print("Connected:", client)
print("Scheduler:", client.scheduler_info()["address"])
print("Workers:", list(client.scheduler_info()["workers"].keys()))

Connected: <Client: 'tcp://10.92.1.34:8786' processes=2 threads=4, memory=15.52 GiB>
Scheduler: tcp://10.92.1.34:8786
Workers: ['tcp://10.92.1.34:39991', 'tcp://10.92.1.34:45745']
